# SWIN Transformer using MONAI

## Check GPU

In [1]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA compiled:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: C:\Users\user\anaconda3\envs\monai\python.exe
PyTorch: 2.11.0+cu128
CUDA compiled: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 5070


# Random seed

In [2]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# More deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed: {SEED}")

Random seed: 42


## Imports

In [15]:
import logging
import numpy as np
import os
from pathlib import Path
import sys
import tempfile
import torch
import ignite

from monai.config import print_config
from monai.data import DataLoader
from monai.engines import SupervisedTrainer, SupervisedEvaluator
from monai.handlers import (
    StatsHandler,
    TensorBoardStatsHandler,
    ValidationHandler,
    CheckpointSaver,
    CheckpointLoader,
    ClassificationSaver,
)
from monai.handlers.utils import from_engine
from monai.inferers import SimpleInferer
from monai.transforms import LoadImageD, EnsureChannelFirstD, ScaleIntensityD, Compose, AsDiscreted
from monai.data import Dataset
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Resized,
    ScaleIntensityd,
    EnsureTyped,
)

print_config()

MONAI version: 1.6.0
Numpy version: 2.2.6
Pytorch version: 2.11.0+cu128
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: eccefc57550b111ed781d82249dfe77872a0e918
MONAI __file__: C:\Users\<username>\anaconda3\envs\monai\lib\site-packages\monai\__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.4.2
scikit-image version: NOT INSTALLED or UNKNOWN VERSION.
scipy version: 1.15.3
Pillow version: 12.3.0
Tensorboard version: 2.21.0
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.26.0+cu128
tqdm version: 4.70.1
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.2.2
pandas version: 2.3.3
einops version: 0.8.2
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.


## Loading Dataset

In [4]:
DATA_DIR = Path(
    r"..\data\brain-tumor-mri-classification\merged_dataset_v3\images"
)

class_names = [
    "glioma",
    "meningioma",
    "no_tumor",
    "pituitary",
]

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

def get_image_files(split):
    split_dir = DATA_DIR / split
    data = []

    for class_name in class_names:
        class_dir = split_dir / class_name

        for image_path in class_dir.iterdir():
            if image_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                data.append(
                    {
                        "image": str(image_path),
                        "label": class_to_idx[class_name],
                    }
                )

    return data

transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Resized(keys=["image"], spatial_size=(224, 224)),
    ScaleIntensityd(keys=["image"]),
    EnsureTyped(keys=["image"]),
])

train_files = get_image_files("train")
val_files = get_image_files("val")
test_files = get_image_files("test")

train_dataset = Dataset(
    data=train_files,
    transform=transforms,
)

val_dataset = Dataset(
    data=val_files,
    transform=transforms,
)

test_dataset = Dataset(
    data=test_files,
    transform=transforms,
)

In [5]:
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

print("First sample:", train_dataset[0])

Train images: 8306
Validation images: 488
Test images: 246
First sample: {'image': metatensor([[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0041, 0.0041, 0.0041],
         [0.0000, 0.0000, 0.0000,  ..., 0.0123, 0.0123, 0.0123],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0082, 0.0041, 0.0041],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0041]],

        [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0041, 0.0041, 0.0041],
         [0.0000, 0.0000, 0.0000,  ..., 0.0123, 0.0123, 0.0123],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0082, 0.0041, 0.0041],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0041]],

        [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  .

In [6]:
sample = train_dataset[0]

print(type(sample))
print(sample.keys())

<class 'dict'>
dict_keys(['image', 'label'])


In [7]:
print("Image Type :", type(sample["image"]))
print("Image Shape:", sample["image"].shape)
print("Image dtype:", sample["image"].dtype)
print("Label:", sample["label"])

Image Type : <class 'monai.data.meta_tensor.MetaTensor'>
Image Shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: 0


## Select subset of dataset

In [8]:
from torch.utils.data import Subset
import random

random.seed(42)

subset_size = 8306

train_indices = random.sample(
    range(len(train_dataset)),
    min(subset_size, len(train_dataset))
)

val_indices = random.sample(
    range(len(val_dataset)),
    min(subset_size, len(val_dataset))
)

test_indices = random.sample(
    range(len(test_dataset)),
    min(subset_size, len(test_dataset))
)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)
test_subset = Subset(test_dataset, test_indices)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Test images:", len(test_subset))

Training images: 8306
Validation images: 488
Test images: 246


## Training the Model

In [19]:
from monai.networks.nets.swin_unetr import SwinTransformer as SwinViT
import torch
import torch.nn as nn

backbone = SwinViT(
    in_chans=3,
    embed_dim=48,             # Corresponds to SwinUNETR's feature_size
    window_size=(7, 7),       # Window size for local self-attention
    patch_size=(2, 2),        # Resolves 2D spatial patches 
    depths=(2, 2, 2, 2),      # Crucial: Use 'depths' instead of 'layers'
    num_heads=(3, 6, 12, 24), # Heads per stage 
    spatial_dims=2            # Force 2D convolutions/attention
)

max_epochs = 50
save_interval = 2

out_dir = "../eval/swin"

device = torch.device("cuda:0")

class SwinClassifier(nn.Module):
    def __init__(self, encoder, num_classes=4, feature_dim=768): # 768 is default final stage dim for embed_dim=48
        super().__init__()
        self.encoder = encoder
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(feature_dim, num_classes)
        
    def forward(self, x):
        # Swin encoder outputs hidden states from each stage
        hidden_states = self.encoder(x)
        # Grab the last stage features
        last_stage_features = hidden_states[-1] # Shape: [Batch, Channel, H_prime, W_prime]
        
        # Pool down and project to your class counts
        pooled = self.pool(last_stage_features).squeeze(-1).squeeze(-1)
        return self.fc(pooled)

model = SwinClassifier(backbone).to(device)

loss_fn = torch.nn.CrossEntropyLoss()

logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
)

evaluator = SupervisedEvaluator(
    device=device,

    val_data_loader=DataLoader(
        val_dataset,
        batch_size=512,
        shuffle=False,
        num_workers=0,
    ),

    network=model,

    inferer=SimpleInferer(),

    key_val_metric={

        "val_accuracy": ignite.metrics.Accuracy(
            output_transform=from_engine(
                ["pred", "label"]
            )
        ),

        "val_loss": ignite.metrics.Loss(
            loss_fn=loss_fn,
            output_transform=from_engine(
                ["pred", "label"]
            ),
        ),
    },

    val_handlers=[

        StatsHandler(
            iteration_log=False,
        ),

        TensorBoardStatsHandler(
            iteration_log=False,
            tag_name="validation",
        ),
    ],
)

trainer = SupervisedTrainer(

    device=device,

    max_epochs=max_epochs,

    train_data_loader=DataLoader(
        train_subset,
        batch_size=32,
        shuffle=True,
        num_workers=0,
    ),

    network=model,

    optimizer=torch.optim.Adam(
        model.parameters(),
        lr=1e-5,
    ),

    loss_function=loss_fn,

    inferer=SimpleInferer(),

    key_train_metric={

        "train_accuracy": ignite.metrics.Accuracy(
            output_transform=from_engine(
                ["pred", "label"]
            )
        ),

        "train_loss": ignite.metrics.Loss(
            loss_fn=loss_fn,
            output_transform=from_engine(
                ["pred", "label"]
            ),
        ),
    },

    train_handlers=[

        ValidationHandler(
            validator=evaluator,
            epoch_level=True,
            interval=1,
        ),

        CheckpointSaver(
            save_dir=out_dir,
            save_dict={
                "model": model
            },
            save_interval=save_interval,
            save_final=True,
            final_filename="monai_swin.pt",
        ),

        TensorBoardStatsHandler(
            tag_name="train",
            output_transform=from_engine(
                ["loss"],
                first=True,
            ),
        ),
    ],
)

In [20]:
from ignite.engine import Events
import datetime

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

@trainer.on(Events.EPOCH_COMPLETED)
def log_epoch_metrics(engine):
    epoch = engine.state.epoch

    train_accuracy = engine.state.metrics.get("train_accuracy")
    train_loss = engine.state.metrics.get("train_loss")

    val_accuracy = evaluator.state.metrics.get("val_accuracy")
    val_loss = evaluator.state.metrics.get("val_loss")

    current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    epoch_duration = engine.state.times.get(Events.EPOCH_COMPLETED.name, 0.0)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)


    print(
        f"\n {current_time} | "
        f"Epoch {epoch}/{max_epochs} | "
        f"Train Accuracy: {train_accuracy} | "
        f"Train Loss: {train_loss} | "
        f"Val Accuracy: {val_accuracy} | "
        f"Val Loss: {val_loss} | "
        f"Time taken: {epoch_duration}\n"
    )

In [21]:
trainer.run()

INFO:ignite.engine.engine.SupervisedTrainer:Engine run resuming from iteration 0, epoch 0 until 50 epochs
ERROR:ignite.engine.engine.SupervisedTrainer:Engine run is terminating due to exception: 
INFO:ignite.engine.engine.SupervisedTrainer:Exception raised, saved the last checkpoint: ../eval/swin\monai_swin.pt


KeyboardInterrupt: 